In [ ]:
pip list

In [10]:
# ============================================
# 1. IMPORTS
# ============================================
import pandas as pd
import numpy as np
import os
import json
import joblib
import sys

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import FunctionTransformer

# ============================================
# 2. CAMINHOS
# ============================================
BASE_PATH = r"C:/Users/Fernanda Pavan/OneDrive/Desktop/Projeto_Risco_Credito"
MODEL_PATH = os.path.join(BASE_PATH, "models")

os.makedirs(MODEL_PATH, exist_ok=True)

# ============================================
# 3. IMPORTAR FEATURE ENGINEERING (SRC)
# ============================================
sys.path.append(BASE_PATH)

from src.features import criar_faixas

# ============================================
# 4. CARREGAR DADOS
# ============================================
df = pd.read_csv(os.path.join(BASE_PATH, "data", "german_credit_data.csv"))

# ============================================
# 5. TRATAMENTO
# ============================================
df['Saving accounts'] = df['Saving accounts'].fillna('little')
df['Checking account'] = df['Checking account'].fillna('little')
df['Saving accounts'] = df['Saving accounts'].replace('quite rich', 'rich')

df.rename(columns={
    'Age': 'Idade',
    'Sex': 'Genero',
    'Job': 'Trabalho',
    'Housing': 'Habitacao',
    'Saving accounts': 'Conta_poupanca',
    'Checking account': 'Conta_corrente',
    'Credit amount': 'Valor_credito',
    'Duration': 'Duracao',
    'Purpose': 'Finalidade',
    'Risk': 'Risco'
}, inplace=True)

df['Risco'] = df['Risco'].map({'good':1, 'bad':0})

# ============================================
# 6. SPLIT
# ============================================
X = df.drop('Risco', axis=1)
y = df['Risco']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# ============================================
# 7. COLUNAS CATEGÓRICAS
# ============================================
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

# ============================================
# 8. PIPELINE
# ============================================
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

pipeline = ImbPipeline(steps=[
    ('features', FunctionTransformer(criar_faixas)),
    ('prep', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('select', SelectKBest(score_func=f_classif, k=10)),
    ('model', LogisticRegression(max_iter=1000))
])

# ============================================
# 9. TREINAR
# ============================================
pipeline.fit(X_train, y_train)

# ============================================
# 10. AVALIAÇÃO
# ============================================
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:,1]

metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "roc_auc": float(roc_auc_score(y_test, y_prob)),
    "classification_report": classification_report(y_test, y_pred, output_dict=True)
}

with open(os.path.join(MODEL_PATH, "metricas.json"), "w") as f:
    json.dump(metrics, f, indent=4)

# ============================================
# 11. RESULTADOS
# ============================================
resultados = X_test.copy()
resultados["y_true"] = y_test
resultados["y_pred"] = y_pred
resultados["prob"] = y_prob

resultados.to_csv(os.path.join(MODEL_PATH, "resultados.csv"), index=False)

# ============================================
# 12. FEATURES SELECIONADAS
# ============================================
ohe = pipeline.named_steps['prep']
features = ohe.get_feature_names_out()

selector = pipeline.named_steps['select']
mask = selector.get_support()

selected_features = features[mask]

with open(os.path.join(MODEL_PATH, "features_selecionadas.json"), "w") as f:
    json.dump(selected_features.tolist(), f, indent=4)

# ============================================
# 13. SALVAR MODELO
# ============================================
joblib.dump(pipeline, os.path.join(MODEL_PATH, "modelo.pkl"))

# ============================================
# FINAL
# ============================================
print("✅ Pipeline de PRODUÇÃO finalizado!")
print(f"📊 Acurácia: {metrics['accuracy']:.4f}")
print(f"📈 ROC AUC: {metrics['roc_auc']:.4f}")
print(f"📁 Salvo em: {MODEL_PATH}")

✅ Pipeline de PRODUÇÃO finalizado!
📊 Acurácia: 0.6300
📈 ROC AUC: 0.6044
📁 Salvo em: C:/Users/Fernanda Pavan/OneDrive/Desktop/Projeto_Risco_Credito\models
